# Egyptian Arabic ASR — Historical Score Verification

## Purpose

This notebook documents the experiment that produced the historical Whisper scores reported in the main **Egyptian Arabic ASR Benchmark** notebook.

Its role is narrow: **provide a reproducible record of how those earlier scores were obtained**. It is not presented as a new benchmark, a comprehensive evaluation, or evidence that the results generalize to every Egyptian Arabic dataset.

The recorded comparison covers:
- `small`
- `large-v3`
- `large-v3 + prompt`
- `egy-finetune` (an additional comparison included in this run)

The original execution outputs are retained so that the reported metrics can be inspected alongside the experiment that generated them.

## What this run actually evaluated

The notebook attempted to load two public datasets. The saved execution output shows that **MGB3 loaded successfully with 60 clips (approximately 24 minutes)**, while both configured MasriSpeech dataset IDs failed to load. Therefore, the recorded scores in this run are **MGB3-only results**, not averages across two datasets.

The models were evaluated on the same loaded MGB3 sample, using the same audio preparation, Arabic normalization, decoding settings, and scoring functions. This makes the model-to-model comparison within this run more controlled.

## How to read this notebook

Follow the cells in order:
1. Load and prepare the labeled MGB3 sample.
2. Normalize reference and hypothesis text consistently.
3. Run each model configuration on the same clips.
4. Calculate WER and CER and inspect the saved result table.

The existing code and recorded outputs are the experiment record. No new model scores are claimed in this documentation.

## 1. Environment setup
Install the dependencies used by the original experiment. The package installation cell is retained as part of the recorded execution workflow.

In [ ]:
!pip install -q faster-whisper jiwer datasets soundfile pandas nvidia-cublas-cu12 nvidia-cudnn-cu12

## 2. GPU and runtime setup

The experiment was run with a CUDA-capable GPU and `faster-whisper` in float16 mode. This cell prepares the CUDA libraries required by the runtime.

The original run used a Colab T4 GPU. Runtime hardware can affect inference time, so timing values such as RTF should not be treated as hardware-independent benchmarks.

In [ ]:
import ctypes, glob, os, torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then restart."

import nvidia.cublas, nvidia.cudnn
for pkg, pattern in [(nvidia.cublas, "libcublasLt.so*"),
                     (nvidia.cublas, "libcublas.so*"),
                     (nvidia.cudnn,  "libcudnn*.so*")]:
    libdir = os.path.join(list(pkg.__path__)[0], "lib")
    for so in sorted(glob.glob(os.path.join(libdir, pattern))):
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError as e:
            print("skipped", os.path.basename(so), "->", e)

import ctranslate2
print("GPU:", torch.cuda.get_device_name(0))
print("CTranslate2 sees", ctranslate2.get_cuda_device_count(), "GPU(s)")

GPU: Tesla T4
CTranslate2 sees 1 GPU(s)


## 3. Experiment configuration

The settings below define the sample size, dataset candidates, fine-tuned model candidate, and the four experiment labels.

**Important scope note:** although MGB3 and MasriSpeech are configured here, the saved output confirms that only MGB3 loaded successfully. The scores preserved in this notebook therefore use 60 MGB3 clips. The failed MasriSpeech load is documented in the output and is not silently treated as a second evaluation set.

All model configurations in this run use the same successfully loaded clips. `large-v3 + prompt` differs from `large-v3` by the supplied Egyptian Arabic initial prompt.

In [ ]:
N_CLIPS = 60
SEED = 42

# Each dataset has a list of candidate Hub names; the first one that loads is used.
DATASETS = {
    "MGB3":  dict(names=["severo/Egyptian-ASR-MGB-3", "MightyStudent/Egyptian-ASR-MGB-3"],
                  split="train", text_col="sentence"),
    "Masri": dict(names=["NightPrince/MasriSpeech-ASR-Finetuning", "NightPrince/MasriSpeech-Full"],
                  split="validation", text_col="transcription"),
}

# Egyptian fine-tunes in Transformers format (the old MAdel121 repo is a SpeechBrain
# checkpoint, which CTranslate2 cannot convert). Tried in order; first success wins.
EGY_CANDIDATES = [
    ("xlpy/whisper-small-egyptian",     "openai/whisper-small"),
    ("alexstokes/whisper-small-eg2",    "openai/whisper-small"),
]
EGY_DIR = "whisper-egy-ct2"               # converted copy for faster-whisper
EGY_LABEL = "egy-finetune"
EGY_PROMPT = "ازيك عامل ايه؟ ده مش هينفع كده خالص، يلا بينا نشوف"  # nudges Whisper toward dialect spelling

EXPERIMENTS = [
    # label,                 model,       extra transcribe() options
    ("small",                "small",     {}),
    ("large-v3",             "large-v3",  {}),
    ("large-v3 + prompt",    "large-v3",  {"initial_prompt": EGY_PROMPT}),
    (EGY_LABEL,              EGY_DIR,     {}),
]


## 4. Arabic text normalization

Egyptian Arabic does not have one consistently applied spelling convention. Before calculating normalized WER and CER, the experiment applies the same normalization function to references and model hypotheses.

The function removes Arabic diacritics and tatweel, standardizes selected alef/ya/ta-marbuta forms, converts Arabic-Indic digits, removes punctuation, and collapses whitespace.

Consequently, the reported `norm_WER` and `norm_CER` measure errors **after this specific normalization**. They should not be described as raw-text metrics.

In [ ]:

import re

_DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u0640]")  # diacritics + tatweel

def normalize_ar(t):
    t = _DIAC.sub("", str(t).lower())
    t = re.sub("[إأآٱ]", "ا", t)
    t = t.replace("ى", "ي").replace("ة", "ه")
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

print(normalize_ar("أَهْلاً بِكُمْ، مَدْرَسَة!"))   # expected: اهلا بكم مدرسه

اهلا بكم مدرسه


## 5. Scoring definitions

The scoring helper calculates the following metrics from the normalized references and hypotheses:

- **`norm_WER`** — word error rate after Arabic text normalization.
- **`norm_CER`** — character error rate after Arabic text normalization.
- **`median_CER`** — median per-clip CER; this describes the typical clip and is distinct from the aggregate CER.
- **`loops`** — count of hypotheses that exceed the reference-length threshold defined in the code.

The table also includes **`raw_WER`**, calculated before text normalization, and **`RTF`**, inference time divided by total audio duration. Lower WER/CER is better. RTF is a timing measure, not an accuracy measure.

For matching the historical benchmark's normalized CER figures, use the `norm_CER` column—not `median_CER`.

In [ ]:
import jiwer, numpy as np

def safe_cer(r, h):
    try:
        return jiwer.cer(r, h)
    except Exception:
        return 1.0

def score(refs, hyps):
    """refs/hyps: lists of already-normalized strings (refs non-empty)."""
    clip_cer = [safe_cer(r, h) for r, h in zip(refs, hyps)]
    loops = sum(len(h.split()) > 3 * len(r.split()) + 5 for r, h in zip(refs, hyps))
    return {
        "norm_WER":   round(jiwer.wer(refs, hyps), 3),
        "norm_CER":   round(jiwer.cer(refs, hyps), 3),
        "median_CER": round(float(np.median(clip_cer)), 3),
        "loops":      int(loops),
    }, clip_cer

## 6. Load the evaluation sample

The loader streams and shuffles the configured dataset, selects up to `N_CLIPS`, converts audio to 16 kHz mono, and normalizes each reference transcript.

**Recorded outcome:** the saved output shows 60 MGB3 clips loaded successfully (about 24.0 minutes). The two MasriSpeech candidates were unavailable, so no MasriSpeech results were produced in this run.

This matters when interpreting the scores: they are based on one 60-clip MGB3 sample, not a multi-dataset evaluation.

In [ ]:
from datasets import load_dataset
from scipy.signal import resample_poly
from math import gcd
from tqdm.auto import tqdm

def to_16k_mono(audio_obj):
    """Works with both the new decoder object and the old dict format."""
    if hasattr(audio_obj, "get_all_samples"):
        d = audio_obj.get_all_samples()
        x, sr = d.data.numpy(), d.sample_rate
    else:
        x, sr = np.asarray(audio_obj["array"]), audio_obj["sampling_rate"]
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:                       # stereo -> mono (handles channel-first or last)
        x = x.mean(axis=0 if x.shape[0] <= x.shape[1] else 1)
    x = x.squeeze().astype(np.float32)
    if sr != 16000:
        g = gcd(sr, 16000)
        x = resample_poly(x, 16000 // g, sr // g).astype(np.float32)
    return x

def load_clips(cfg, n, seed=SEED):
    last_err = None
    for name in cfg["names"]:
        try:
            ds = load_dataset(name, split=cfg["split"], streaming=True)
            ds = ds.shuffle(seed=seed, buffer_size=300).take(n)
            clips = []
            for s in tqdm(ds, total=n, desc=name.split("/")[-1]):
                ref = normalize_ar(s[cfg["text_col"]])
                if not ref:
                    continue
                clips.append({"audio": to_16k_mono(s["audio"]), "ref": ref,
                              "raw_ref": s[cfg["text_col"]]})
            print("  using", name)
            return clips
        except Exception as e:
            last_err = e
            print(f"  {name} failed: {type(e).__name__}: {str(e)[:120]}")
    raise last_err

DATA = {}
for key, cfg in DATASETS.items():
    try:
        DATA[key] = load_clips(cfg, N_CLIPS)
        mins = sum(len(c["audio"]) for c in DATA[key]) / 16000 / 60
        print(f"{key}: {len(DATA[key])} clips, {mins:.1f} min")
    except Exception as e:
        print(f"!! could not load {key}: {type(e).__name__}: {e}")

assert DATA, "No dataset loaded. Check the dataset names in the settings cell."

README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

Egyptian-ASR-MGB-3:   0%|          | 0/60 [00:00<?, ?it/s]

  using severo/Egyptian-ASR-MGB-3
MGB3: 60 clips, 24.0 min
  NightPrince/MasriSpeech-ASR-Finetuning failed: DatasetNotFoundError: Dataset 'NightPrince/MasriSpeech-ASR-Finetuning' doesn't exist on the Hub or cannot be accessed.
  NightPrince/MasriSpeech-Full failed: DatasetNotFoundError: Dataset 'NightPrince/MasriSpeech-Full' doesn't exist on the Hub or cannot be accessed.
!! could not load Masri: DatasetNotFoundError: Dataset 'NightPrince/MasriSpeech-Full' doesn't exist on the Hub or cannot be accessed.


## 7. Prepare the Egyptian fine-tuned comparison

This section converts the configured Egyptian fine-tuned checkpoint into the format required by `faster-whisper`. It is included because that model was one of the configurations evaluated in the original run.

The fine-tuned model's score is an additional result from this experiment; it is not one of the three historical Whisper baseline/prompt scores that this notebook is primarily documenting.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download
from ctranslate2.converters import TransformersConverter

egy_ok = os.path.exists(os.path.join(EGY_DIR, "model.bin"))
if not egy_ok:
    for hf_id, base in EGY_CANDIDATES:
        try:
            shutil.rmtree(EGY_DIR, ignore_errors=True)
            TransformersConverter(hf_id).convert(EGY_DIR, quantization="float16", force=True)
            # tokenizer / feature-extractor config from the base model it was fine-tuned from
            for f in ["tokenizer.json", "preprocessor_config.json"]:
                shutil.copy(hf_hub_download(base, f), os.path.join(EGY_DIR, f))
            print("Egyptian model ready:", hf_id, os.listdir(EGY_DIR))
            EGY_LABEL_FULL = hf_id
            egy_ok = True
            break
        except Exception as e:
            print(f"!! {hf_id} failed: {type(e).__name__}: {str(e)[:200]}")

if not egy_ok:
    print("   No Egyptian model could be converted; that experiment will be skipped.")
    EXPERIMENTS = [x for x in EXPERIMENTS if x[1] != EGY_DIR]


config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/4.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Egyptian model ready: xlpy/whisper-small-egyptian ['preprocessor_config.json', 'model.bin', 'config.json', 'vocabulary.json', 'tokenizer.json']


## 8. Run the recorded model comparisons

Each model is loaded in turn and transcribes the same loaded dataset clips. Decoding uses Arabic transcription, beam size 5, and `condition_on_previous_text=False`; the prompted experiment additionally supplies the Egyptian Arabic initial prompt configured above.

The code calculates the metrics using the shared normalization and scoring functions, then writes `results.csv` and `clip_results.csv`.

The saved execution output is important provenance: it records the model labels and the resulting WER/CER values from this run.

In [ ]:
import gc, time, pandas as pd
from faster_whisper import WhisperModel

def transcribe_clips(model, clips, **opts):
    hyps, infer = [], 0.0
    for c in tqdm(clips, leave=False):
        t0 = time.perf_counter()
        segs, _ = model.transcribe(c["audio"], language="ar", task="transcribe",
                                   beam_size=5, condition_on_previous_text=False, **opts)
        text = " ".join(s.text.strip() for s in segs)
        infer += time.perf_counter() - t0
        hyps.append(text)
    return hyps, infer

rows, clip_rows = [], []
for model_id in dict.fromkeys(x[1] for x in EXPERIMENTS):          # unique models, in order
    print(f"\n=== loading {model_id}")
    model = WhisperModel(model_id, device="cuda", compute_type="float16")
    for label, mid, opts in [x for x in EXPERIMENTS if x[1] == model_id]:
        for dname, clips in DATA.items():
            hyps, infer = transcribe_clips(model, clips, **opts)
            n_hyps = [normalize_ar(h) for h in hyps]
            refs   = [c["ref"] for c in clips]
            metrics, clip_cer = score(refs, n_hyps)
            audio_sec = sum(len(c["audio"]) for c in clips) / 16000
            rows.append({"dataset": dname, "experiment": label,
                         "raw_WER": round(jiwer.wer([c["raw_ref"] for c in clips], hyps), 3),
                         **metrics, "RTF": round(infer / audio_sec, 3)})
            for c, h, cc in zip(clips, n_hyps, clip_cer):
                clip_rows.append({"dataset": dname, "experiment": label,
                                  "ref": c["ref"], "hyp": h, "CER": cc})
            print(f"{label:22s} {dname:6s} WER {metrics['norm_WER']:.3f}  CER {metrics['norm_CER']:.3f}")
    del model; gc.collect(); torch.cuda.empty_cache()

results = pd.DataFrame(rows)
clip_df = pd.DataFrame(clip_rows)
results.to_csv("results.csv", index=False)
clip_df.to_csv("clip_results.csv", index=False)
print("\nSaved results.csv and clip_results.csv")


=== loading small


  0%|          | 0/60 [00:00<?, ?it/s]

small                  MGB3   WER 0.595  CER 0.290

=== loading large-v3


  0%|          | 0/60 [00:00<?, ?it/s]

large-v3               MGB3   WER 0.363  CER 0.191


  0%|          | 0/60 [00:00<?, ?it/s]

large-v3 + prompt      MGB3   WER 0.356  CER 0.187

=== loading whisper-egy-ct2


  0%|          | 0/60 [00:00<?, ?it/s]

egy-finetune           MGB3   WER 0.543  CER 0.264

Saved results.csv and clip_results.csv


## 9. Recorded results

The table below is the original saved result output from this notebook. It reports one dataset (`MGB3`) and four model configurations. **Lower normalized WER and CER indicate fewer recognition errors.**

The three historical normalized CER values can be traced directly to the `norm_CER` column:
- `small`: **0.290**
- `large-v3`: **0.191**
- `large-v3 + prompt`: **0.187**

The corresponding normalized WER values are 0.595, 0.363, and 0.356. The table also includes `median_CER`; that is a separate per-clip statistic and must not be substituted for `norm_CER`.

These are the values recorded by this particular 60-clip MGB3 run. They are not claims of performance on the unavailable MasriSpeech split or on unseen recordings.

In [ ]:
pd.set_option("display.width", 200)
display(results.sort_values(["dataset", "norm_CER"]))
print("Average across datasets:")
display(results.groupby("experiment")[["norm_WER", "norm_CER", "median_CER", "RTF"]].mean().round(3).sort_values("norm_CER"))

,dataset,experiment,raw_WER,norm_WER,norm_CER,median_CER,loops,RTF
2,MGB3,large-v3 + prompt,0.502,0.356,0.187,0.155,0,0.169
1,MGB3,large-v3,0.501,0.363,0.191,0.159,0,0.156
3,MGB3,egy-finetune,0.618,0.543,0.264,0.216,0,0.075
0,MGB3,small,0.653,0.595,0.290,0.250,0,0.058


Average across datasets:


,norm_WER,norm_CER,median_CER,RTF
experiment,,,,
large-v3 + prompt,0.356,0.187,0.155,0.169
large-v3,0.363,0.191,0.159,0.156
egy-finetune,0.543,0.264,0.216,0.075
small,0.595,0.290,0.250,0.058


## 10. Per-clip error inspection

This section displays the highest-error clips for the experiment with the lowest average normalized CER in the recorded results. It provides a qualitative check of model hypotheses against the dataset references.

A high error can reflect a recognition mistake, a reference-transcript issue, or both. The examples are useful for inspecting possible causes, but they do not independently establish that a reference is wrong.

In [ ]:
best = results.groupby("experiment")["norm_CER"].mean().idxmin()
print("Best experiment:", best, "\n")
worst = clip_df[clip_df.experiment == best].sort_values("CER", ascending=False).head(8)
for _, r in worst.iterrows():
    print(f"[{r.dataset}] CER {r.CER:.2f}\n  REF: {r.ref}\n  HYP: {r.hyp}\n")

Best experiment: large-v3 + prompt 

[MGB3] CER 0.63
  REF: والالوان كمان بتاعتها لطيفه يعني طالع دلوقتي النبيتي طبعا مكتسح الدنيا كلها النبيتي في كل حاجه النبيتي اللي هو الماروني شويه ده اه هي تقريبا من وراه شكل الطرحه
  HYP: والالوان كمان بتاعتها لطيفه يعني طالعه دلوقتي النبيتي طبعا مقتصح الدنيا كلها النبيتي كل حاجه النبيتي اللي هو الماروني شويه والاحمرات طبعا بدرجاتها نتيجه بعد دكي بس تلفيا شويه بالكورسي عشان نشوف كمان من ورا الشكل عامل ازاي دارينا من ورا شكل الطرحه

[MGB3] CER 0.62
  REF: ف برضو هيتم ليها هيتم فيها بعض الزيارات ده هياخد مننا وقت تلات تيام ال ال الاولانين يعني اليوم الرابع احنا سيبينه للشباب دول لما تروح جينيف في سيرن ناوي علي ايه هل مثلا بتفكر ان انت تاخد notesلكل حاجه وتيجي تطبقها هنا ولا بس مجرد انها زياره هتشوف بس هما بيفكروا ازاي وترجع تاني
  HYP: كده في الديتكتوس فبرضو حيتم فيها بعض الزيارات ده حياخد مننا ال 3 ايام الاولمين يعني الروم الرابع احنا سايبينه للشباب دول يروحوا وينفوا في جيناف ما يتبوش مننا يعني وان شاء الله نرجع يوم عشان يتفسحوا بقي مش كله شغل ودرا

## 11. Reproducibility record and scope

The original run saves:
- `results.csv` — aggregate metrics by dataset and experiment.
- `clip_results.csv` — normalized reference, hypothesis, and per-clip CER records.

The notebook retains the original code and saved outputs associated with the historical scores. To reproduce the experiment, rerun the notebook in a compatible environment with access to the referenced Hugging Face datasets and model checkpoints. Dataset availability, package versions, and GPU hardware may have changed since the recorded run.

**Interpretation limit:** the three historical scores are supported here as outputs of this recorded MGB3 experiment on the selected 60-clip sample. This is evidence of where the numbers came from; it is not a claim of an exhaustive benchmark or a guarantee that a fresh run will reproduce identical values.